# Project8 v0.5 — ba pilot đơn giản từ BEST v0.5

T4 ×2, imgsz 1280, batch 32; 12.000 train/2.500 val, không dùng 500 test.

L0A0: LR 0,001/augmentation gốc; L1A0: LR 0,0005/augmentation gốc; L0A1: LR 0,001/augmentation giảm nhẹ. Mỗi lượt 2 epoch.

Schedule chung: lrf=1, warmup_epochs=0,25, warmup_bias_lr=0; là đối chứng ngắn mới, khác schedule lịch sử. Người dùng cho phép pilot, full visual review chưa hoàn tất. Gắn archive dataset cũ và Input riêng tư chứa BEST v0.5 trước Run All. Hash checkpoint được kiểm tra bắt buộc.


In [ ]:
# 1. Ba pilot don gian: moi luot 2 epoch, cung BEST v0.5
START_TRAINING = True
ALLOW_UNREVIEWED_PILOT = True  # user authorized exploratory pilots; full review remains incomplete
ARCHIVE_INPUT = ''
DATASET_INPUT = ''
CHECKPOINT_INPUT = ''  # search attached notebook output by SHA256
GPU_IDS = '0,1'
EPOCHS = 2
PATIENCE = 0
IMGSZ = 1280
BATCH = 32
CLOSE_MOSAIC = 0
SMOKE_ONLY = False
RUN_ONE_EPOCH_SMOKE = False
OUTPUT_ROOT = '/kaggle/working/BTL-AI-Camera-v05-pilots'
OLD_NAMES = ['person', 'table', 'chair', 'laptop', 'cell phone', 'backpack', 'book', 'cup']
NAMES = OLD_NAMES[:-1] + ['bottle']
EXPECTED_IMAGES = {'train': 12000, 'val': 2500, 'test': 500}
EXPECTED_BOXES = {'train': 33272, 'val': 8359, 'test': 1645}
BEST_SHA256 = '92adfeb2516861a6958e8adc4a15cf764faa3956078da1ebc81db1fbb7014e31'
MANIFEST_SHA256 = '22a0140ab5e4d454d7e0c46ab24145c6d52d8b949db67ab8142fdd3bb8bb9ccd'
CHECKSUMS_SHA256 = '53a664318ad67bb64f6c894c21e94a6543382b5b9cf859e5ca9d4e283db4a77e'
PORTABLE_YAML_SHA256 = '4b26114dfaf683ce88b3b159cdec3f37f30f9b8168fd260c6cc1a977d718a122'
BASE_AUG = dict(mosaic=0.5, scale=0.5, translate=0.1, fliplr=0.5,
                hsv_h=0.015, hsv_s=0.7, hsv_v=0.4)
LIGHT_AUG = dict(mosaic=0.2, scale=0.25, translate=0.05, fliplr=0.25,
                 hsv_h=0.0075, hsv_s=0.35, hsv_v=0.2)
EXPERIMENTS = [dict(label='L0A0_control', lr0=0.001, aug=BASE_AUG),
               dict(label='L1A0_half_lr', lr0=0.0005, aug=BASE_AUG),
               dict(label='L0A1_light_aug', lr0=0.001, aug=LIGHT_AUG)]
# Shared short-run schedule: flat post-warmup LR, warmup 0.25 epoch, bias LR 0.
# This control is a new 2-epoch baseline; historical 5-epoch schedule differs.


In [ ]:
# 2. Giữ phiên bản Ultralytics đã dùng tạo checkpoint; không cài requirements CPU local
import subprocess, sys, torch
before_torch = (torch.__version__, torch.version.cuda)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.165'], check=True)
import ultralytics, yaml
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset
from pathlib import Path
import csv, json, hashlib, math, shutil, datetime, gc, zipfile
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
assert ultralytics.__version__ == '8.4.165'
assert before_torch == (torch.__version__, torch.version.cuda), 'Torch/CUDA bị thay đổi; cần reset môi trường.'
print('Ultralytics:', ultralytics.__version__, '| Torch/CUDA:', before_torch)
print('GPU:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# 3. Dataset archive giu nguyen; checkpoint train la BEST v0.5 theo SHA256
INPUT_ROOT = Path('/kaggle/input')
def file_sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def safe_path(base, relative):
    path = Path(relative)
    if path.is_absolute() or '..' in path.parts:
        raise ValueError(f'Đường dẫn không an toàn: {relative}')
    result = (base / path).resolve()
    if not result.is_relative_to(base.resolve()):
        raise ValueError(f'Đường dẫn ra ngoài dataset: {relative}')
    return result

# Một file Input; giữ đuôi .zip.bin để Kaggle không tự bung lúc upload.
import stat, time
ARCHIVE_SHA256 = '236972ca3e6142d977a29068cf217dc5312c8fb870c8094a98c4960386a0eb5b'
ARCHIVE_NAME = 'project8_v05_trainval_best_v04_20261008.zip.bin'
ARCHIVE_ROOT = None
archive_paths = [Path(ARCHIVE_INPUT)] if ARCHIVE_INPUT else sorted(INPUT_ROOT.rglob(ARCHIVE_NAME))
if len(archive_paths) > 1:
    raise RuntimeError('Có nhiều archive; đặt ARCHIVE_INPUT cho một file cụ thể.')
if archive_paths:
    archive_path = archive_paths[0]
    print('ARCHIVE_HASH_START:', archive_path, flush=True)
    started = time.perf_counter()
    assert file_sha(archive_path) == ARCHIVE_SHA256, 'Sai SHA256 archive train/val + BEST v0.4'
    ARCHIVE_ROOT = Path('/tmp/project8_v05_15k_runtime') / ('archive_' + ARCHIVE_SHA256[:16])
    ARCHIVE_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        infos = archive.infolist()
        member_names = [info.filename for info in infos]
        assert len(member_names) == len(set(member_names)) == 29005, 'Archive thiếu/trùng file'
        manifest_bytes = archive.read('project8_v05/manifest.csv')
        assert hashlib.sha256(manifest_bytes).hexdigest() == MANIFEST_SHA256
        archive_rows = list(csv.DictReader(manifest_bytes.decode('utf-8-sig').splitlines()))
        assert len(archive_rows) == 15000
        expected_members = {
            'project8_v05/' + row[key]
            for row in archive_rows if row['split'] in {'train', 'val'}
            for key in ('image_path', 'label_path')
        } | {
            'project8_v05/manifest.csv', 'project8_v05/checksums.json',
            'project8_v05/data.yaml', 'project8_v05/training_readiness.json',
            'BEST_Project8_v04_20261008.pt',
        }
        assert set(member_names) == expected_members, 'Archive có file dư/thiếu hoặc test media'
        assert shutil.disk_usage(ARCHIVE_ROOT).free > sum(info.file_size for info in infos) + 512*1024**2, 'Thiếu ổ tạm để giải nén'
        # Kiểm toàn bộ đường dẫn trước khi ghi bất kỳ file nào.
        for info in infos:
            assert '\\' not in info.filename and not info.is_dir()
            assert not stat.S_ISLNK(info.external_attr >> 16), 'Không nhận symlink'
            safe_path(ARCHIVE_ROOT, info.filename)
        print('ARCHIVE_HASH_PASS: 29005 files; extract locally', flush=True)
        for index, info in enumerate(infos, 1):
            destination = safe_path(ARCHIVE_ROOT, info.filename)
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as read, destination.open('wb') as write:
                shutil.copyfileobj(read, write, length=1024*1024)
            assert destination.stat().st_size == info.file_size
            if index % 2000 == 0 or index == len(infos):
                print('UNZIP:', index, '/', len(infos), '| seconds:', round(time.perf_counter()-started, 1), flush=True)
    INPUT_ROOT = ARCHIVE_ROOT
    print('ARCHIVE_READY:', INPUT_ROOT, '| seconds:', round(time.perf_counter()-started, 1), flush=True)
else:
    print('Không có archive; dùng Input đã bung sẵn.', flush=True)

manifests = [Path(DATASET_INPUT) / 'manifest.csv'] if DATASET_INPUT else sorted(INPUT_ROOT.rglob('manifest.csv'))
valid_roots = [p.parent for p in manifests if p.is_file() and file_sha(p) == MANIFEST_SHA256 and (p.parent/'images/train').is_dir()]
if not valid_roots:
    raise FileNotFoundError('Chưa gắn đúng dataset v0.5 15k. Add Input dataset một file .zip.bin hoặc dataset v0.5 đã bung sẵn.')
if len(valid_roots) != 1:
    raise RuntimeError('Có nhiều bản dataset 15k; đặt DATASET_INPUT cho một bản cụ thể.')
SOURCE_DATA = valid_roots[0]
assert file_sha(SOURCE_DATA / 'checksums.json') == CHECKSUMS_SHA256
checksums = json.loads((SOURCE_DATA / 'checksums.json').read_text(encoding='utf-8'))
with (SOURCE_DATA / 'manifest.csv').open(encoding='utf-8-sig', newline='') as stream:
    rows = list(csv.DictReader(stream))
assert len(rows) == 15000 and Counter(r['split'] for r in rows) == EXPECTED_IMAGES
cfg = yaml.safe_load((SOURCE_DATA/'data.yaml').read_text(encoding='utf-8'))
names = cfg['names'] if isinstance(cfg['names'], list) else [cfg['names'][i] for i in range(8)]
assert names == NAMES and cfg['nc'] == 8
assert len({r['pixel_sha256'] for r in rows}) == 15000
expected_files = {r[k] for r in rows for k in ('image_path','label_path')}
assert set(checksums) == expected_files | {'manifest.csv','data.yaml'} and len(checksums) == 30002
assert checksums['manifest.csv'] == MANIFEST_SHA256
# Export YAML is portable; source data.yaml digest remains in the historical release checksums.
assert file_sha(SOURCE_DATA/'data.yaml') == PORTABLE_YAML_SHA256
assert cfg.get('train')=='images/train' and cfg.get('val')=='images/val' and not cfg.get('test')

weight_paths = [Path(CHECKPOINT_INPUT)] if CHECKPOINT_INPUT else sorted(set(Path('/kaggle/input').rglob('*.pt')) | set(INPUT_ROOT.rglob('*.pt')))
matched_weights = [p for p in weight_paths if p.is_file() and file_sha(p) == BEST_SHA256]
if not matched_weights:
    raise FileNotFoundError('Thieu BEST v0.5 dung SHA256; Add Input output notebook v0.5.')
initial = matched_weights[0]  # các bản còn lại, nếu có, đều cùng byte/hash
model_check = YOLO(str(initial))
assert model_check.task == 'detect' and [model_check.names[i] for i in range(8)] == NAMES
print('Dataset:', SOURCE_DATA, '| split:', dict(Counter(r['split'] for r in rows)))
print('BEST v0.5:', initial, '| SHA256:', file_sha(initial))
print('ID 7 bottle gồm cup+bottle; giữ nguyên 8 ID từ best v0.4.')
del model_check; gc.collect()
readiness = json.loads((SOURCE_DATA/'training_readiness.json').read_text(encoding='utf-8'))
assert readiness['manifest_sha256'] == MANIFEST_SHA256
TRAIN_GATE = (readiness.get('training_ready') is True and readiness.get('full_visual_review_complete') is True and all(r['review_stage']=='full_review_complete' for r in rows))
print('Full visual review gate:', TRAIN_GATE)

assert {split:sum(int(r['boxes']) for r in rows if r['split']==split) for split in EXPECTED_IMAGES} == EXPECTED_BOXES


In [ ]:
# 4. Stage và kiểm SHA256/tọa độ toàn bộ train+val; test không vào thư mục train
WORK = Path('/tmp/project8_v05_15k_runtime')
DATA_DIR = SOURCE_DATA if ARCHIVE_ROOT is not None else WORK / 'dataset'
DATA_DIR.mkdir(parents=True, exist_ok=True)
train_val = [r for r in rows if r['split'] in {'train','val'}]

def stage_record(row):
    for key, hash_key in [('image_path','image_sha256'),('label_path','label_sha256')]:
        src = safe_path(SOURCE_DATA, row[key]); dst = safe_path(DATA_DIR, row[key])
        assert checksums[row[key]] == row[hash_key]
        assert file_sha(src) == row[hash_key], f'File nguồn bị đổi: {src}'
        if src != dst:
            dst.parent.mkdir(parents=True, exist_ok=True)
            if not dst.is_file() or file_sha(dst) != row[hash_key]:
                # Input đã bung: đọc/ghi thường để tránh sendfile trên mount.
                with src.open('rb') as read, dst.open('wb') as write:
                    shutil.copyfileobj(read, write, length=1024*1024)
            assert file_sha(dst) == row[hash_key], dst
        # Archive: file đã trên ổ tạm và vừa được hash; không chép/hash lần hai.
    counts = Counter()
    for line in (DATA_DIR/row['label_path']).read_text(encoding='utf-8').splitlines():
        values = list(map(float, line.split()))
        assert len(values)==5 and all(math.isfinite(v) for v in values)
        cls,x,y,w,h = values
        assert cls.is_integer() and 0<=cls<8 and 0<=x<=1 and 0<=y<=1 and 0<w<=1 and 0<h<=1
        counts[int(cls)] += 1
    assert sum(counts.values()) == int(row['boxes']) and counts
    return counts

class_counts = Counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    for i, counts in enumerate(pool.map(stage_record, train_val), 1):
        class_counts.update(counts)
        if i % 2000 == 0 or i == len(train_val):
            print('Stage/hash/labels:', i, '/', len(train_val), flush=True)
assert sum(class_counts.values()) == 41631
assert not (DATA_DIR/'images/test').exists() and not (DATA_DIR/'labels/test').exists()
actual_files = {p.relative_to(DATA_DIR).as_posix() for folder in ['images','labels'] for p in (DATA_DIR/folder).rglob('*') if p.is_file() and not p.name.endswith('.cache')}
assert actual_files == {r[k] for r in train_val for k in ('image_path','label_path')}, 'Có file dư trong scratch; dùng session mới.'
DATA_YAML = WORK/'training_data.yaml'
DATA_YAML.write_text(yaml.safe_dump({'path':str(DATA_DIR.resolve()), 'train':'images/train', 'val':'images/val', 'nc':8, 'names':dict(enumerate(NAMES))},sort_keys=False),encoding='utf-8')
resolved = check_det_dataset(str(DATA_YAML),autodownload=False)
assert not resolved.get('test')
for split in ['train','val']:
    assert Path(resolved[split]) == (DATA_DIR/'images'/split).resolve()
OUT = Path(OUTPUT_ROOT); OUT.mkdir(parents=True, exist_ok=True)
PREPARED = {'dataset_manifest_sha256':MANIFEST_SHA256,'checkpoint_sha256':BEST_SHA256,'all_input_images':EXPECTED_IMAGES,'training_images':len(train_val),'training_boxes':41631,'classes':NAMES,'class_id_7':'cups+bottles by user request','resume':False,'cls_remap':False}
(OUT/'prepared.json').write_text(json.dumps(PREPARED,indent=2),encoding='utf-8')
print('PREPARED: 12.000 train + 2.500 val; 500 test giữ riêng. Chưa train.')
print('Box train+val:', {NAMES[i]:class_counts[i] for i in range(8)})

In [ ]:
# 5. Xem ảnh và bounding box của từng lớp trước train
from PIL import Image, ImageDraw
from IPython.display import display
canvas = Image.new('RGB',(4*320,2*260),'#eeeeee')
for cls,name in enumerate(NAMES):
    row = next(r for r in train_val if any(int(line.split()[0])==cls for line in (DATA_DIR/r['label_path']).read_text().splitlines()))
    with Image.open(DATA_DIR/row['image_path']) as im:
        im = im.convert('RGB'); im.thumbnail((312,220))
        draw = ImageDraw.Draw(im)
        for line in (DATA_DIR/row['label_path']).read_text().splitlines():
            c,x,y,w,h = map(float,line.split())
            if int(c)==cls:
                draw.rectangle(((x-w/2)*im.width,(y-h/2)*im.height,(x+w/2)*im.width,(y+h/2)*im.height),outline='lime',width=2)
        x0,y0 = (cls%4)*320,(cls//4)*260
        canvas.paste(im,(x0+(320-im.width)//2,y0))
        ImageDraw.Draw(canvas).text((x0+4,y0+228),f'{cls}: {name}',fill='black')
preview = OUT/'dataset_boxes_preview.jpg'; canvas.save(preview,quality=90)
display(canvas)

In [ ]:
# 6. Train 3 pilots x 2 epochs from identical BEST v0.5; CLI handles DDP.
TRAIN_COMPLETED = False
PILOT_RUNS = []
if not START_TRAINING:
    print('Configuration prepared; training disabled.')
else:
    if not TRAIN_GATE and not ALLOW_UNREVIEWED_PILOT:
        raise RuntimeError('Full visual review incomplete; exploratory pilot authorization required.')
    assert torch.cuda.is_available(), 'Select GPU T4 x2; no CPU fallback.'
    count = torch.cuda.device_count()
    ids = [int(x) for x in GPU_IDS.split(',')]
    assert ids == [0, 1] and count >= 2
    device = ','.join(map(str,ids))
    batch = BATCH
    assert batch == 32 and batch % len(ids) == 0
    assert EPOCHS == 2 and PATIENCE == 0 and CLOSE_MOSAIC == 0
    assert IMGSZ == 1280 and len(EXPERIMENTS) == 3
    assert file_sha(initial) == BEST_SHA256
    yolo_cli = shutil.which('yolo')
    assert yolo_cli, 'Run install cell first.'
    stamp = datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=7))).strftime('%Y%m%d_%H%M%S')
    RUNS = OUT / 'kaggle_runs'
    for experiment in EXPERIMENTS:
        # Preserve verified complete runs; retry incomplete branches from identical initial weights.
        complete = []
        for candidate in sorted(RUNS.glob('project8_v05_' + experiment['label'] + '_e2_*')):
            metadata = candidate / 'pilot_metadata.json'
            if not metadata.is_file():
                continue
            saved = json.loads(metadata.read_text(encoding='utf-8'))
            assert saved['parent_sha256'] == BEST_SHA256
            assert saved['label'] == experiment['label'] and saved['completed_epochs'] == 2
            cfg = saved['config']
            expected = dict(lr0=experiment['lr0'], imgsz=1280, batch=32, epochs=2,
                optimizer='MuSGD', lrf=1.0, warmup_epochs=0.25, warmup_bias_lr=0.0,
                seed=42, deterministic=True, resume=False, nms=False, close_mosaic=0,
                **experiment['aug'])
            assert all(cfg[k] == v for k,v in expected.items()), 'Recovered config mismatch'
            with (candidate/'results.csv').open(newline='') as stream:
                recovered_rows = [{k.strip():v.strip() for k,v in r.items()} for r in csv.DictReader(stream)]
            assert len(recovered_rows) == 2
            recovered_epoch = int(float(max(recovered_rows,key=lambda r:float(r['metrics/mAP50(B)']))['epoch']))
            assert saved['selected_epoch_mAP50'] == recovered_epoch
            checkpoint = candidate/'weights'/f'epoch{recovered_epoch-1}.pt'
            assert checkpoint.is_file() and file_sha(checkpoint) == saved['selected_sha256']
            saved['run_dir'] = str(candidate)
            saved['selected_checkpoint'] = str(checkpoint)
            complete.append(saved)
        if complete:
            PILOT_RUNS.append(complete[-1])
            (OUT/'pilot_progress.json').write_text(json.dumps(PILOT_RUNS,indent=2),encoding='utf-8')
            print('PILOT_REUSED_COMPLETE:',experiment['label'],complete[-1]['run_dir'],flush=True)
            continue
        if experiment['label'] == 'L0A0_control':
            print('CONTROL_CHECKPOINT_UNAVAILABLE: historical log retained; uniform re-evaluation excludes control.',flush=True)
            continue
        run_name = 'project8_v05_' + experiment['label'] + '_e2_' + stamp
        train_config = dict(model=str(initial), data=str(DATA_YAML), project=str(RUNS), name=run_name,
            epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=batch, device=device,
            workers=2, cache=False, fraction=1.0, optimizer='MuSGD', lr0=experiment['lr0'],
            lrf=1.0, cos_lr=False, warmup_epochs=0.25, warmup_bias_lr=0.0, warmup_momentum=0.8,
            momentum=0.937, weight_decay=0.0005, seed=42, deterministic=True,
            flipud=0.0, degrees=0.0, shear=0.0, perspective=0.0, bgr=0.0,
            mixup=0.0, cutmix=0.0, close_mosaic=0, multi_scale=False,
            save=True, save_period=1, resume=False, cls_remap=False, val=True,
            nms=False, max_det=300, conf=0.001, plots=True, exist_ok=False, **experiment['aug'])
        config_path = OUT / (experiment['label'] + '_config.yaml')
        config_path.write_text(yaml.safe_dump(train_config,sort_keys=False),encoding='utf-8')
        print('PILOT_START:',experiment['label'],'| parent:',file_sha(initial),'| epochs:',EPOCHS,flush=True)
        subprocess.run([yolo_cli,'detect','train',f'cfg={config_path}'],check=True,cwd=str(OUT))
        run_dir = RUNS / run_name
        with (run_dir/'results.csv').open(newline='') as stream:
            epoch_rows = [{k.strip():v.strip() for k,v in r.items()} for r in csv.DictReader(stream)]
        assert len(epoch_rows) == 2, 'Pilot must finish exactly 2 epochs.'
        target_row = max(epoch_rows,key=lambda r:float(r['metrics/mAP50(B)']))
        target_epoch = int(float(target_row['epoch']))
        # Ultralytics checkpoint filename is zero based; CSV epoch is one based.
        selected = run_dir/'weights'/f'epoch{target_epoch-1}.pt'
        assert selected.is_file() and (run_dir/'weights/best.pt').is_file() and (run_dir/'weights/last.pt').is_file()
        record = dict(label=experiment['label'],run_dir=str(run_dir),config=train_config,
            completed_epochs=2,selected_epoch_mAP50=target_epoch,selected_checkpoint=str(selected),
            parent_sha256=BEST_SHA256,selected_sha256=file_sha(selected),epoch_metrics=epoch_rows,
            full_visual_review_complete=bool(TRAIN_GATE),status='train_complete_eval_pending')
        (run_dir/'pilot_metadata.json').write_text(json.dumps(record,indent=2),encoding='utf-8')
        PILOT_RUNS.append(record)
        (OUT/'pilot_progress.json').write_text(json.dumps(PILOT_RUNS,indent=2),encoding='utf-8')
        plots = sorted(run_dir.glob('train_batch*.jpg'))
        if plots: display(Image.open(plots[0]))
        print('PILOT_TRAIN_COMPLETE:',experiment['label'],'| best AP50 epoch:',target_epoch,flush=True)
    TRAIN_COMPLETED = True
    print('Available pilots complete; evaluation next. Missing historical control is excluded. No test set used.')


In [ ]:
# 7. Old smoke branch replaced by the three controlled pilots above.
print('Pilot status:', TRAIN_COMPLETED, '| runs:', len(PILOT_RUNS))


In [ ]:
# 8. Evaluate initial BEST v0.5 and selected AP50 epoch on identical 2500 val images.
if not TRAIN_COMPLETED:
    print('Train pilots first; no evaluation result yet.')
else:
    EVAL_PROTOCOL = dict(data=str(DATA_YAML),split='val',imgsz=IMGSZ,batch=8,device=0,
        conf=0.001,iou=0.7,nms=False,max_det=300,augment=False,rect=True,
        quantize=16,workers=2,plots=True)
    def evaluate_checkpoint(checkpoint,label):
        evaluated = YOLO(str(checkpoint))
        assert evaluated.task == 'detect' and [evaluated.names[i] for i in range(8)] == NAMES
        metrics = evaluated.val(**EVAL_PROTOCOL,project=str(OUT/'comparison'),name=label,exist_ok=False)
        assert evaluated.model.end2end, 'NMS-free YOLO26 evaluation required.'
        record = dict(checkpoint_sha256=file_sha(checkpoint),dataset_manifest_sha256=MANIFEST_SHA256,
            protocol=EVAL_PROTOCOL,mAP50=float(metrics.box.map50),mAP50_95=float(metrics.box.map),
            precision=float(metrics.box.mp),recall=float(metrics.box.mr),per_class={})
        for j,cls in enumerate(metrics.box.ap_class_index):
            record['per_class'][NAMES[int(cls)]] = dict(AP50=float(metrics.box.ap50[j]),
                AP50_95=float(metrics.box.ap[j]),precision=float(metrics.box.p[j]),recall=float(metrics.box.r[j]))
        (OUT/(label+'_val_metrics.json')).write_text(json.dumps(record,indent=2),encoding='utf-8')
        del evaluated; gc.collect(); torch.cuda.empty_cache()
        return record
    pilot_comparison = dict(parent_sha256=BEST_SHA256,manifest_sha256=MANIFEST_SHA256,
        full_visual_review_complete=bool(TRAIN_GATE),protocol=EVAL_PROTOCOL,
        unavailable_runs=[e['label'] for e in EXPERIMENTS if e['label'] not in {r['label'] for r in PILOT_RUNS}],runs={})
    baseline = evaluate_checkpoint(initial,'BEST_v05_before')
    pilot_comparison['runs']['BEST_v05_before'] = baseline
    for record in PILOT_RUNS:
        measured = evaluate_checkpoint(Path(record['selected_checkpoint']),record['label'])
        measured['delta_mAP50_percentage_points'] = 100*(measured['mAP50']-baseline['mAP50'])
        measured['selected_epoch'] = record['selected_epoch_mAP50']
        pilot_comparison['runs'][record['label']] = measured
        (OUT/'pilot_comparison.json').write_text(json.dumps(pilot_comparison,indent=2),encoding='utf-8')
        print(record['label'],'| mAP50:',measured['mAP50'],'| delta pp:',measured['delta_mAP50_percentage_points'],flush=True)
    pilot_comparison['status'] = 'complete'
    (OUT/'pilot_comparison.json').write_text(json.dumps(pilot_comparison,indent=2),encoding='utf-8')
    import pandas as pd
    display(pd.DataFrame({label:{'mAP50 %':100*r['mAP50'],'mAP50-95 %':100*r['mAP50_95'],
        'delta AP50 pp':r.get('delta_mAP50_percentage_points',0)} for label,r in pilot_comparison['runs'].items()}).T)
    print('Two epochs are exploratory; confirm promising settings in a longer matched run.')


In [ ]:
# 9. Preserve three pilot runs and comparison in one verified ZIP.
if not TRAIN_COMPLETED or 'pilot_comparison' not in globals() or pilot_comparison.get('status') != 'complete':
    print('Pilots/evaluation incomplete; intermediate files remain in OUTPUT_ROOT.')
else:
    sources = {p.relative_to(OUT).as_posix():p for p in OUT.rglob('*')
        if p.is_file() and p.suffix.lower() in {'.pt','.csv','.yaml','.json','.jpg','.png'}
        and (p.suffix.lower() != '.pt' or 'kaggle_runs' in p.relative_to(OUT).parts)}
    files = {name:dict(sha256=file_sha(p),bytes=p.stat().st_size) for name,p in sources.items()}
    result_zip = OUT/('project8_v05_three_pilots_e2_'+stamp+'_result.zip')
    with zipfile.ZipFile(result_zip,'w',zipfile.ZIP_DEFLATED) as archive:
        for name,p in sources.items(): archive.write(p,name)
        archive.writestr('FILES.json',json.dumps(dict(schema_version=3,files=files),indent=2))
    with zipfile.ZipFile(result_zip) as archive: assert archive.testzip() is None
    print('PILOTS_COMPLETE:',result_zip,'| SHA256:',file_sha(result_zip),flush=True)
    print('Save Version to preserve outputs before the session ends.')


## Đọc kết quả

Đọc pilot_comparison.json, results.csv và pilot_metadata.json. Epoch chọn theo mAP50 có thể khác best.pt chọn theo mAP50–95. Hai epoch chỉ sàng lọc; xác nhận cấu hình có triển vọng bằng lượt dài hơn. ZIP cuối chứa kết quả và trọng số. Save Version để bảo toàn output.


## Hai lượt ngắn bổ sung theo yêu cầu ngày 09/10/2026
Mục tiêu trước mắt 80% mAP50. Chỉ L1A1 và L2A0, mỗi lượt 2 epoch; dùng kết quả/preflight từ các cell trước. Full visual review vẫn chưa hoàn tất.


In [ ]:
# Authorized bounded follow-up: only two NEW runs, 2 epochs each, identical initial checkpoint.
import datetime, subprocess, shutil, csv, json, gc, zipfile
from pathlib import Path
import torch, yaml
from ultralytics import YOLO
from ultralytics.cfg import get_cfg
assert START_TRAINING and ALLOW_UNREVIEWED_PILOT
assert torch.cuda.device_count() >= 2 and file_sha(initial) == BEST_SHA256
assert PREPARED['dataset_manifest_sha256'] == MANIFEST_SHA256
assert len([r for r in rows if r['split']=='train']) == 12000
assert len([r for r in rows if r['split']=='val']) == 2500
old_out = Path('/kaggle/working/BTL-AI-Camera-v05-pilots')
previous = json.loads((old_out/'pilot_comparison.json').read_text())
assert previous['status']=='complete' and previous['parent_sha256']==BEST_SHA256
assert previous['manifest_sha256']==MANIFEST_SHA256
follow_stamp = datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=7))).strftime('%Y%m%d_%H%M%S')
follow_out = old_out/('quick_followup_'+follow_stamp)
follow_out.mkdir(exist_ok=False)
follow_runs = follow_out/'kaggle_runs'
template = yaml.safe_load((old_out/'L1A0_half_lr_config.yaml').read_text())
assert template['epochs']==2 and template['imgsz']==1280 and template['batch']==32
assert template['optimizer']=='MuSGD' and template['nms'] is False
assert template['resume'] is False and template['seed']==42
follow_experiments=[dict(label='L1A1_half_lr_light_aug',lr0=.0005,aug=LIGHT_AUG),dict(label='L2A0_low_lr_base_aug',lr0=.0001,aug=BASE_AUG)]
follow_protocol=dict(previous['protocol'])
follow_result={'status':'running','target_mAP50':.80,'parent_sha256':BEST_SHA256,'manifest_sha256':MANIFEST_SHA256,'full_visual_review_complete':False,'protocol':follow_protocol,'baseline':previous['runs']['BEST_v05_before'],'runs':{}}
for exp in follow_experiments:
    cfg=dict(template)
    cfg.update(model=str(initial),data=str(DATA_YAML),project=str(follow_runs),name=exp['label']+'_e2_'+follow_stamp,lr0=exp['lr0'],**exp['aug'])
    assert cfg['epochs']==2 and cfg['patience']==0 and cfg['lrf']==1 and cfg['close_mosaic']==0
    get_cfg(overrides=cfg)
    config_path=follow_out/(exp['label']+'_config.yaml')
    config_path.write_text(yaml.safe_dump(cfg,sort_keys=False))
    (follow_out/'comparison.json').write_text(json.dumps(follow_result,indent=2))
    print('QUICK_FOLLOWUP_START',exp['label'],'epochs=2','LR=',exp['lr0'],'parent=',BEST_SHA256,flush=True)
    subprocess.run([shutil.which('yolo'),'detect','train',f'cfg={config_path}'],check=True,cwd=str(follow_out))
    run_dir=follow_runs/cfg['name']
    epoch_rows=[{k.strip():v.strip() for k,v in r.items()} for r in csv.DictReader((run_dir/'results.csv').open())]
    assert len(epoch_rows)==2
    selected_epoch=int(float(max(epoch_rows,key=lambda r:float(r['metrics/mAP50(B)']))['epoch']))
    selected=run_dir/'weights'/f'epoch{selected_epoch-1}.pt'
    assert selected.is_file()
    model=YOLO(str(selected))
    measured=model.val(**follow_protocol,project=str(follow_out/'comparison'),name=exp['label'],exist_ok=False)
    assert model.model.end2end
    record={'selected_epoch':selected_epoch,'selected_checkpoint':str(selected),'checkpoint_sha256':file_sha(selected),'config':cfg,'epoch_metrics':epoch_rows,'mAP50':float(measured.box.map50),'mAP50_95':float(measured.box.map),'precision':float(measured.box.mp),'recall':float(measured.box.mr),'per_class':{}}
    for j,cls in enumerate(measured.box.ap_class_index):
        record['per_class'][NAMES[int(cls)]]={'AP50':float(measured.box.ap50[j]),'AP50_95':float(measured.box.ap[j])}
    record['delta_mAP50_percentage_points']=100*(record['mAP50']-follow_result['baseline']['mAP50'])
    follow_result['runs'][exp['label']]=record
    (run_dir/'pilot_metadata.json').write_text(json.dumps(record,indent=2))
    (follow_out/'comparison.json').write_text(json.dumps(follow_result,indent=2))
    print('QUICK_FOLLOWUP_COMPLETE',exp['label'],record['mAP50'],record['delta_mAP50_percentage_points'],flush=True)
    del model;gc.collect();torch.cuda.empty_cache()
follow_result['status']='complete'
(follow_out/'comparison.json').write_text(json.dumps(follow_result,indent=2))
follow_zip=old_out/('project8_quick_followups_'+follow_stamp+'.zip')
with zipfile.ZipFile(follow_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(follow_out.rglob('*')):
        if p.is_file():z.write(p,p.relative_to(follow_out).as_posix())
with zipfile.ZipFile(follow_zip) as z:assert z.testzip() is None
print('ALL_QUICK_FOLLOWUPS_COMPLETE',str(follow_out),'ZIP=',str(follow_zip),flush=True)
print(json.dumps({k:{'mAP50':v['mAP50'],'delta_pp':v['delta_mAP50_percentage_points']} for k,v in follow_result['runs'].items()}),flush=True)


In [ ]:
# Confirm extra NMS with the exact GPU FP16/batch8 protocol; head stays end-to-end.
from ultralytics.models.yolo.detect import DetectionValidator
from ultralytics.utils.nms import TorchNMS
class ExtraClassNMSValidator(DetectionValidator):
    def postprocess(self, preds):
        items=super().postprocess(preds)
        assert self.end2end and self.args.nms is False
        for item in items:
            selected=[]
            for cls in item['cls'].unique():
                ids=(item['cls']==cls).nonzero(as_tuple=False).flatten()
                keep=TorchNMS.nms(item['bboxes'][ids],item['conf'][ids],.6)
                selected.append(ids[keep])
            if selected:
                keep=torch.cat(selected)
                keep=keep[item['conf'][keep].argsort(descending=True)]
                for key in ['bboxes','conf','cls','extra']:item[key]=item[key][keep]
        return items
assert follow_result['status']=='complete'
all_candidates={'BEST_v05_before':dict(checkpoint=str(initial),mAP50=previous['runs']['BEST_v05_before']['mAP50'])}
for label,r in previous['runs'].items():
    if label!='BEST_v05_before':
        old_record=next(x for x in PILOT_RUNS if x['label']==label)
        all_candidates[label]=dict(checkpoint=old_record['selected_checkpoint'],mAP50=r['mAP50'])
for label,r in follow_result['runs'].items():all_candidates[label]=dict(checkpoint=r['selected_checkpoint'],mAP50=r['mAP50'])
winner=max(all_candidates,key=lambda k:all_candidates[k]['mAP50'])
labels=list(dict.fromkeys(['BEST_v05_before',winner]))
nms_confirmation={'status':'running','protocol':follow_protocol,'extra_class_NMS_iou':.6,'head':'same end2end; nms=False','selection':'threshold selected on 300-val pilot, checkpoint selected on same validation; not independent test','winner_without_extra_NMS':winner,'runs':{}}
for label in labels:
    checkpoint=Path(all_candidates[label]['checkpoint']);assert checkpoint.is_file()
    model=YOLO(str(checkpoint))
    result=model.val(validator=ExtraClassNMSValidator,**follow_protocol,project=str(follow_out/'extra_nms_comparison'),name=label,exist_ok=False)
    assert model.model.end2end
    record=dict(checkpoint_sha256=file_sha(checkpoint),mAP50=float(result.box.map50),mAP50_95=float(result.box.map),precision=float(result.box.mp),recall=float(result.box.mr),without_extra_NMS_mAP50=all_candidates[label]['mAP50'],per_class={})
    record['delta_vs_same_checkpoint_percentage_points']=100*(record['mAP50']-record['without_extra_NMS_mAP50'])
    for j,cls in enumerate(result.box.ap_class_index):record['per_class'][NAMES[int(cls)]]=dict(AP50=float(result.box.ap50[j]),AP50_95=float(result.box.ap[j]))
    nms_confirmation['runs'][label]=record
    (follow_out/'extra_nms_confirmation.json').write_text(json.dumps(nms_confirmation,indent=2))
    print('EXTRA_NMS_CONFIRMED',label,record['mAP50'],record['delta_vs_same_checkpoint_percentage_points'],flush=True)
    del model;gc.collect();torch.cuda.empty_cache()
nms_confirmation['status']='complete'
(follow_out/'extra_nms_confirmation.json').write_text(json.dumps(nms_confirmation,indent=2))
final_zip=old_out/('project8_quick_followups_with_nms_'+follow_stamp+'.zip')
with zipfile.ZipFile(final_zip,'w',zipfile.ZIP_DEFLATED) as z:
    files={}
    for p in sorted(follow_out.rglob('*')):
        if p.is_file():
            rel=p.relative_to(follow_out).as_posix();z.write(p,rel);files[rel]=dict(bytes=p.stat().st_size,sha256=file_sha(p))
    z.writestr('FILES.json',json.dumps(files,indent=2))
with zipfile.ZipFile(final_zip) as z:assert z.testzip() is None
print('QUICK_RESULTS_PRESERVED',str(final_zip),file_sha(final_zip),flush=True)
print(json.dumps(nms_confirmation),flush=True)
